In [4]:
# Celda 1 — Verificar GPU e instalar dependencias
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ninguna'}")

!pip install ultralytics roboflow -q
print("✅ Dependencias instaladas")

GPU disponible: True
GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 107.1 MB/s eta 0:00:00
✅ Dependencias instaladas


In [5]:
# Celda 2 — Conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = "/content/drive/MyDrive/logo-detection"
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"✅ Drive conectado: {DRIVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive conectado: /content/drive/MyDrive/logo-detection


In [6]:
# Celda 3 — Descargar dataset desde Roboflow
from roboflow import Roboflow

ROBOFLOW_API_KEY = "tlCulKdO0462FO1mTlrw"  # ← pon tu key aquí
DATASET_PATH = "/content/dataset"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("dimitar-dimitrov-qnnci").project("logo-detection-clean")
version = project.version(3)
dataset = version.download("yolov8", location=DATASET_PATH)

print(f"✅ Dataset descargado en: {DATASET_PATH}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/dataset in yolov8:: 100%|██████████| 34400/34400 [00:05<00:00, 5973.54it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset descargado en: /content/dataset


In [7]:
# Celda 4 — Fine-tuning YOLOv8
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data=f"{DATASET_PATH}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=DRIVE_PATH,
    name="logo-detection-v1",
    patience=10,
    save=True,
    plots=True,
)

print(f"\n✅ Entrenamiento completado")
print(f"mAP@50: {results.results_dict['metrics/mAP50(B)']:.3f}")

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=logo-detection-v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

In [8]:
# Celda final — Verificar pesos guardados
import os

best_pt = "/content/drive/MyDrive/logo-detection/logo-detection-v1/weights/best.pt"
last_pt = "/content/drive/MyDrive/logo-detection/logo-detection-v1/weights/last.pt"

print(f"best.pt: {'✅ existe' if os.path.exists(best_pt) else '❌ no encontrado'}")
print(f"last.pt: {'✅ existe' if os.path.exists(last_pt) else '❌ no encontrado'}")
print(f"\nRuta para el equipo: MyDrive/logo-detection/logo-detection-v1/weights/best.pt")

best.pt: ✅ existe
last.pt: ✅ existe

Ruta para el equipo: MyDrive/logo-detection/logo-detection-v1/weights/best.pt
